http://nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

# Import Library

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType

# Create SparkSession

In [3]:
spark = SparkSession.builder.appName("Bronze to Silver").getOrCreate()
print("SparkSession created successfully.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/26 21:09:56 WARN Utils: Your hostname, Anle-Lenovo, resolves to a loopback address: 127.0.1.1; using 192.168.1.23 instead (on interface wlo1)
26/06/26 21:09:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 21:09:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/26 21:09:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession created successfully.


In [ ]:
input_path = "../data/bronze/yellow_tripdata_2016-02.csv"
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(path=input_path)
)

---
# Cleaning

## Drop None, Drop Duplicate, Casting type

In [5]:
def clean_data(df):
    """Clean and standardize raw trip data."""

    # Drop null, drop duplicates
    df_clean = df.dropna().dropDuplicates()

    # Data type casting
    df_clean = (
        df_clean
        # Vendor information
        .withColumn(
            "vendor_id",
            F.col("VendorID").cast(IntegerType()),
        )
        # Pickup and dropoff timestamps
        .withColumn(
            "pickup_datetime",
            F.col("tpep_pickup_datetime").cast(TimestampType()),
        )
        .withColumn(
            "dropoff_datetime",
            F.col("tpep_dropoff_datetime").cast(TimestampType()),
        )
        # Trip details
        .withColumn(
            "passenger_count",
            F.col("passenger_count").cast(IntegerType()),
        )
        .withColumn(
            "trip_distance_miles",
            F.col("trip_distance").cast(DoubleType()),
        )
        .withColumn(
            "rate_code_id",
            F.col("RateCodeID").cast(IntegerType()),
        )
        .withColumn(
            "store_and_fwd_flag",
            F.col("store_and_fwd_flag"),
        )
        # Pickup and dropoff coordinates
        .withColumn(
            "pickup_longitude",
            F.col("pickup_longitude").cast(DoubleType()),
        )
        .withColumn(
            "pickup_latitude",
            F.col("pickup_latitude").cast(DoubleType()),
        )
        .withColumn(
            "dropoff_longitude",
            F.col("dropoff_longitude").cast(DoubleType()),
        )
        .withColumn(
            "dropoff_latitude",
            F.col("dropoff_latitude").cast(DoubleType()),
        )
        # Payment information
        .withColumn(
            "payment_type",
            F.col("payment_type").cast(IntegerType()),
        )
        # Fare and charges
        .withColumn(
            "fare_amount",
            F.col("fare_amount").cast(DoubleType()),
        )
        .withColumn(
            "extra_amount",
            F.col("extra").cast(DoubleType()),
        )
        .withColumn(
            "mta_tax_amount",
            F.col("mta_tax").cast(DoubleType()),
        )
        .withColumn(
            "tip_amount",
            F.col("tip_amount").cast(DoubleType()),
        )
        .withColumn(
            "tolls_amount",
            F.col("tolls_amount").cast(DoubleType()),
        )
        .withColumn(
            "improvement_surcharge",
            F.col("improvement_surcharge").cast(DoubleType()),
        )
        .withColumn(
            "total_amount",
            F.col("total_amount").cast(DoubleType()),
        )
        # Remove original columns after renaming
        .drop(
            "VendorID",
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance",
            "extra",
            "mta_tax",
            "RateCodeID",
        )
    )

    return df_clean

# Save to parquet file

In [ ]:
# silver_df = clean_data(df)
# silver_df.write.mode("overwrite").parquet("../data/silver/data_demo.parquet")

26/06/26 21:10:39 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/26 21:10:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/26 21:10:59 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
